# Real holo-density fusion into pretrained FuncBind

This report compares **original pretrained FuncBind** against the same frozen
FuncBind model conditioned by a residual derived from an aligned real 2Fo-Fc
holo map. The selected case is CrossDocked test target **69: 5MGL / ligand
7MU**.

> **Interpretation boundary:** this is a single-target proof of concept. The
holo map contains the crystallographic ligand, the small density adapter is
overfit on this target, and the displayed molecule from each method is selected
by oracle latent MSE. It tests whether this density representation can steer the
existing decoder; it does not establish held-out generation performance.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

search_roots = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    root for root in search_roots
    if (root / 'exps' / 'density_fusion').exists()
)
RESULT_DIR = (
    PROJECT_ROOT / 'exps' / 'density_fusion'
    / 'holo_5mgl_target69_20260728'
)
with (RESULT_DIR / 'result.json').open() as handle:
    report = json.load(handle)
report['status'], report['target']


## Quantitative result

| Method | Occupancy MSE | mIoU | Pred. atoms | Matched | Precision | Recall | F1 |
|---|---:|---:|---:|---:|---:|---:|---:|
| Original FuncBind | 3.368e-05 | 0.3750 | 11 | 8 | 0.727 | 0.727 | 0.727 |
| Holo-density adapter | 1.834e-05 | 0.5711 | 10 | 10 | 1.000 | 0.909 | 0.952 |

The comparison uses 8 paired chains, the same
initial latent per pair, and restored sampler RNG state. The table reports the
oracle-best chain for each method.


![Metric summary](figures/holo_density_metric_summary.png)


## What the real density looks like

The crop is a 16 Å cube at 0.25 Å resolution, aligned to FuncBind's centered
pocket frame.

![Density slices](figures/holo_density_slices.png)


## 3D occupancy comparison

The first panel plots the real holo-density points at ≥1.5 local σ with the
reference atoms. The remaining panels use the same element colors and an
occupancy threshold of 0.1.

![3D occupancy comparison](figures/holo_density_occupancy_3d.png)


## Paired diffusion behavior

![Paired latent MSE](figures/holo_density_paired_latent_mse.png)


## Adapter fit

Only the 1,118,720-parameter density
adapter was trained; the two FuncBind checkpoints and the VoxBind density
encoder were frozen.

![Adapter training](figures/holo_density_adapter_training.png)


In [ ]:
def load_cloud(path):
    with np.load(path) as data:
        return {key: data[key] for key in data.files}

clouds = {
    'Reference': load_cloud(
        RESULT_DIR / 'reference_occupancy_points.npz'
    ),
    'Original FuncBind': load_cloud(
        RESULT_DIR / 'original_funcbind' / 'occupancy_points.npz'
    ),
    'Holo-density FuncBind': load_cloud(
        RESULT_DIR / 'holo_density_funcbind' / 'occupancy_points.npz'
    ),
    'Real holo density': load_cloud(
        RESULT_DIR / 'density_points.npz'
    ),
}
{
    name: cloud['coordinates'].shape[0]
    for name, cloud in clouds.items()
}


## What this result does—and does not—show

- It directly tests density-side conditioning while leaving the original
  ligand INR decoder intact.
- The unknown ligand atom channels supplied to the density encoder are zero;
  the ligand signal comes from the real holo map.
- A better density-conditioned curve or occupancy fit is evidence that this
  representation can steer FuncBind on this target.
- It cannot distinguish density information from single-target adapter
  memorization. The next valid experiment is to train one adapter on training
  complexes with density and evaluate it, without target fitting or oracle
  selection, on held-out maps including apo maps.

**That control has since been run — see `apo_density_funcbind_report.ipynb`.**
Repeating this experiment on the same map with the ligand erased reproduces the
improvement below (mIoU 0.525 vs 0.571, F1 0.957 vs 0.952), so the gain shown
here comes from fitting the adapter to this one target, not from reading the
ligand out of the density.
